In [5]:
import os
import sys
import platform
import time
import tensorflow as tf
from tensorflow.keras import mixed_precision

print("--- INÍCIO: Validação Total do Ambiente de Deep Learning ---")
print(f"Data e Hora da Verificação: {time.ctime()}")
print("-" * 50)

# 1. Configuração de Logs do TensorFlow (já no docker-compose)
# Reconfirmando aqui para visibilidade no notebook
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
print("***Nível de log do TensorFlow definido para: WARNING e ERROR (INFO oculto).***")

# 2. Verificação de Mixed Precision
# Esta linha deve ser executada antes de qualquer criação de modelo no TensorFlow
mixed_precision.set_global_policy('mixed_float16')
print(f"\n***Mixed Precision ativado. Política atual: {mixed_precision.global_policy().name}.***")
if mixed_precision.global_policy().name == 'mixed_float16':
    print("  - Aceleração via float16 está configurada e pronta para sua GPU NVIDIA.***\n")
else:
    print("  - Aviso: Mixed Precision não ativado. Verifique a configuração.***\n")

print("\n***--- Informações do Ambiente ---***")
print(f"***Versão do TensorFlow: {tf.__version__}***")
print(f"***Versão do Python: {platform.python_version()}***")
print(f"***Sistema Operacional (Dentro do Container): {platform.system()} {platform.release()}***\n")

--- INÍCIO: Validação Total do Ambiente de Deep Learning ---
Data e Hora da Verificação: Sun Jun  1 15:06:10 2025
--------------------------------------------------
***Nível de log do TensorFlow definido para: WARNING e ERROR (INFO oculto).***

***Mixed Precision ativado. Política atual: mixed_float16.***
  - Aceleração via float16 está configurada e pronta para sua GPU NVIDIA.***


***--- Informações do Ambiente ---***
***Versão do TensorFlow: 2.17.0***
***Versão do Python: 3.12.3***
***Sistema Operacional (Dentro do Container): Linux 5.15.153.1-microsoft-standard-WSL2***



In [6]:
print("\n--- 3. Verificação do Interpretador Python ---")
python_executable_path = sys.executable
print(f"Caminho do Interpretador Python em uso: {python_executable_path}")

expected_path = "/usr/bin/python"
if python_executable_path == expected_path:
    print(f"  - Sucesso: O interpretador em uso é o esperado ({expected_path}).")
else:
    print(f"  - Aviso: O interpretador em uso NÃO é o esperado. Esperado: {expected_path}, Atual: {python_executable_path}")
    print("    Verifique a configuração em .devcontainer/devcontainer.json.")


--- 3. Verificação do Interpretador Python ---
Caminho do Interpretador Python em uso: /usr/bin/python
  - Sucesso: O interpretador em uso é o esperado (/usr/bin/python).


In [7]:
print("\n--- 4. Verificação de Acessibilidade da GPU ---")
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"GPU(s) Encontrada(s) pelo TensorFlow: {gpus}")
    for i, gpu in enumerate(gpus):
        print(f"  - GPU {i}: Nome: {gpu.name}")
        try:
            details = tf.config.experimental.get_device_details(gpu)
            print(f"    - Nome do Dispositivo: {details.get('device_name', 'N/A')}")
            print(f"    - Bus ID: {details.get('physical_device_desc', 'N/A')}")
            print(f"    - Capacidade de Computação: {details.get('compute_capability', 'N/A')}")
        except Exception as e:
            print(f"    - Aviso: Não foi possível obter detalhes adicionais da GPU: {e}")

    # Tenta rodar nvidia-smi para mais detalhes do driver e CUDA
    print("\n  - Detalhes do Driver NVIDIA (via nvidia-smi, se disponível):")
    try:
        nvidia_smi_output = os.popen('nvidia-smi').read()
        print(nvidia_smi_output)
        if "NVIDIA-SMI" in nvidia_smi_output:
            driver_version_line = [line for line in nvidia_smi_output.split('\n') if 'Driver Version:' in line]
            if driver_version_line:
                print(f"    - Versão do Driver (Host/WSL2): {driver_version_line[0].split('Driver Version:')[1].split(' ')[1]}")
                print(f"    - Versão CUDA (Host/WSL2): {driver_version_line[0].split('CUDA Version:')[1].split(' ')[1]}")
            print("  - Sucesso: Comunicação com o driver NVIDIA estabelecida.")
        else:
            print("  - Aviso: 'nvidia-smi' não retornou a saída esperada. Pode não estar no PATH.")
    except Exception as e:
        print(f"  - Aviso: Não foi possível executar 'nvidia-smi': {e}")

    print("\n  - Status Final da GPU: Sucesso. GPU(s) detectada(s) e pronta(s) para uso.")
else:
    print("Status Final da GPU: FALHA. Nenhuma GPU detectada pelo TensorFlow.")
    print("  - Verifique drivers, Docker Desktop, integração WSL2 e configurações.")
    print("--- FIM DO TESTE: GPU NÃO DETECTADA ---")
    # exit() # Comentar para não parar o notebook, se quiser que ele continue rodando


--- 4. Verificação de Acessibilidade da GPU ---
GPU(s) Encontrada(s) pelo TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  - GPU 0: Nome: /physical_device:GPU:0
    - Nome do Dispositivo: NVIDIA GeForce RTX 3060 Ti
    - Bus ID: N/A
    - Capacidade de Computação: (8, 6)

  - Detalhes do Driver NVIDIA (via nvidia-smi, se disponível):
Sun Jun  1 15:06:10 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.04              Driver Version: 576.52         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+====

I0000 00:00:1748790371.170604    8887 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [8]:
print("\n--- 5. Teste de Desempenho da GPU (Provas da Performance!) ---")

# Tamanho da matriz para o teste de performance
# Aumentando o tamanho da matriz para um teste mais longo na CPU
matrix_size = 16000 # <-- ALTERAÇÃO AQUI!

# Teste na GPU
print(f"Executando cálculo intensivo (multiplicação de matrizes {matrix_size}x{matrix_size}) na GPU...")
with tf.device('/GPU:0'):
    a_gpu = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)
    b_gpu = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)

    # Primeira execução (warm-up) - geralmente mais lenta devido a inicialização
    _ = tf.matmul(a_gpu, b_gpu) 
    
    start_time_gpu = time.time()
    c_gpu = tf.matmul(a_gpu, b_gpu)
    end_time_gpu = time.time()
    
    gpu_time = end_time_gpu - start_time_gpu
    print(f"  - Tempo de execução na GPU: {gpu_time:.4f} segundos.")

# Teste na CPU
print(f"Executando o mesmo cálculo na CPU para comparação...")
with tf.device('/CPU:0'):
    a_cpu = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)
    b_cpu = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)
    
    # Primeira execução na CPU (warm-up)
    _ = tf.matmul(a_cpu, b_cpu)

    start_time_cpu = time.time()
    c_cpu = tf.matmul(a_cpu, b_cpu)
    end_time_cpu = time.time()
    
    cpu_time = end_time_cpu - start_time_cpu
    print(f"  - Tempo de execução na CPU: {cpu_time:.4f} segundos.")

if gpu_time < cpu_time:
    speedup = cpu_time / gpu_time
    print(f"\nResultado da Performance: GPU está {speedup:.2f}x MAIS RÁPIDA que a CPU!")
else:
    print("\nResultado da Performance: A GPU não está mais rápida que a CPU no teste de matrizes.")
    print("  - Isso pode indicar um problema de configuração ou que o teste não é adequado.")

print("\n--- FIM: Validação Total do Ambiente de Deep Learning ---")

# Mensagem Final para o Usuário
if gpus and gpu_time < cpu_time:
    print("\n\n#####################################################################")
    print("#####################################################################")
    print("##                                                                 ##")
    print("##      AMBIENTE ESTÁ CONFIGURADO E OTIMIZADO!                    ##")
    print("##   A GPU está acessível e rodando a todo vapor.                 ##")
    print("##                                                                 ##")
    print("##                                                                 ##")
    print("#####################################################################")
    print("#####################################################################")
else:
    print("\n\n#####################################################################")
    print("##                                                                 ##")
    print("##    ATENÇÃO: ALGO NÃO ESTÁ 100% OTIMIZADO.                       ##")
    print("##    Verifique os logs acima para identificar a causa.            ##")
    print("##                                                                 ##")
    print("#####################################################################")


--- 5. Teste de Desempenho da GPU (Provas da Performance!) ---
Executando cálculo intensivo (multiplicação de matrizes 16000x16000) na GPU...


ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 